# Case 1: Perfect requests, Perfect Logging, Non-Ambiguous Humans

This will be our baseline case.

![Case 1 Diagram](images/case1diagram.png)

In [33]:
from hitl_telemetry_sim import (
    hitl_system_simulation,
    plot_acceptance_rate_by_request_type,
    plot_acceptance_rate_by_user,
    plot_daily_acceptance,
    run_catboost_auc_slice,
)

## Run the Simulation

In [34]:
sim = hitl_system_simulation(n_components = 6,
                             n_users = 100,
                             seed=7,
                             pr = 1.0, #(Perfect request)
                             p_accept_correct_range = (1.0, 1.0), #(Non-Ambiguous Human)
                             p_accept_incorrect_range = (0.0, 0.0)) #(Non-Ambiguous Human)

In [35]:
sim.run_simulation(
    n_items_per_day=1000,
    simulation_duration=365 + 90,
    sprint_every_days=7,
    sprint_stop_day=365 + 60,
    components_per_sprint=1,
    improvement_rate=0.4,
    min_path_length=2,
)

,Date,User,systemInferencePath,C4,C6,C5,C2,C3,C1,request_type,isAccepted
0,1,u24,"[C1, C2, C3, C4, C5, C6]",True,True,False,True,False,False,Type_1,False
1,1,u83,"[C1, C2, C3, C4, C6]",False,True,NaN,True,False,True,Type_2,False
2,1,u61,"[C2, C4]",False,NaN,NaN,True,NaN,NaN,Type_3,False
3,1,u50,"[C1, C2, C3, C4, C6]",True,True,NaN,True,True,True,Type_2,True
4,1,u26,"[C2, C3, C4, C5, C6]",True,True,False,True,True,NaN,Type_4,False
...,...,...,...,...,...,...,...,...,...,...,...
454995,455,u88,"[C1, C2, C3, C5, C6]",NaN,True,True,True,True,True,Type_20,True
454996,455,u37,"[C1, C2, C4, C6]",True,True,NaN,True,NaN,True,Type_19,True
454997,455,u35,"[C1, C5]",NaN,NaN,True,NaN,NaN,True,Type_26,True
454998,455,u57,"[C2, C3, C4, C5, C6]",True,True,True,True,True,NaN,Type_4,True


## Plotting rate of accepts calculated daily

In [37]:
plot_daily_acceptance(sim.telemetry)

## Plotting rate of accepts by request type

In [39]:
plot_acceptance_rate_by_request_type(sim.telemetry, window=7)

## Plotting rate of accepts by user

In [40]:
plot_acceptance_rate_by_user(sim.telemetry, window=14)

# Use AUC to tells investigate our telemetry data

![AUC 1 Diagram](images/auc1.png)

![AUC 2 Diagram](images/auc2.png)

# AUC tells us if the logged telemetry features contain enough signal to separate outcomes

AUC = 1, strongest feature support for outcome separability

AUC = .5, weakest feature support for outcome separability

## AUC on Users

In [41]:
auc_results = run_catboost_auc_slice(
    telemetry_df=sim.telemetry,
    date_slice=(455 - 30, 455),
    feature_columns=["User"],
)

print(f"Mean AUC: {auc_results['mean_auc']:.3f} ± {auc_results['std_auc']:.3f}")

Mean AUC: 0.500 ± 0.018


## AUC on Request Type

In [42]:
auc_results = run_catboost_auc_slice(
    telemetry_df=sim.telemetry,
    date_slice=(455 - 30, 455),
    feature_columns=["request_type"],
)

print(f"Mean AUC: {auc_results['mean_auc']:.3f} ± {auc_results['std_auc']:.3f}")

Mean AUC: 0.597 ± 0.022


## AUC on Components

In [43]:
request_type_auc = run_catboost_auc_slice(
    telemetry_df=sim.telemetry,
    date_slice=(455 - 30, 455),
    feature_columns=["C1", "C2", "C3", "C4", "C5", "C6"],
)

print(f"Mean AUC: {request_type_auc['mean_auc']:.3f} ± {request_type_auc['std_auc']:.3f}")

Mean AUC: 1.000 ± 0.000
